# 応用編7

V3.3新機能:ACLへのドメインの指定  

このノートブックでは、スマートコントラクト・バージョン3.3から可能になった、ACLへのドメインの指定について説明します。  


## 概要

ACLのうち、visible_to, disclosed_to, accessible_toにはドメインが指定可能です。  
ドメインを指定した場合、権限の判定時点で、そのドメインに含まれるユーザまたはコントラクトに権限が与えられます。  
ドメイン内に一括して権限付与を行いたい場合に適している機能です。

## 準備

In [1]:
var { domain } = await import('../lib/load-config.mjs');
var { adminWallet, rpc, deploySmartContract } = await import('../lib/notebook-util.mjs');

## 説明用スマートコントラクトのデプロイ

サブコントラクトをデプロイします。  
処理の内容は、呼び出したコントラクトのIDを返すというものです。

In [2]:
var subcid = await deploySmartContract(function adv7sub(){
    return getCallerId();
});

上のサブコントラクトを呼び出す親コントラクトを２つデプロイします。  
処理の内容は、呼び出したサブコントラクトの返り値をそのまま返すというものです。  
結果として、自分自身のコントラクトIDを返すことになります。
（この処理は動作の確認を目的としたものであり、実用的な意味はありません。）

In [3]:
var cids = [];
for(var i=0; i<2; i++){
    cids[i] = await deploySmartContract(function(){
        return openContract('__subcid__').call();
    }, {
        name: 'adv7cnt'+i,
        replacers: [['__subcid__', subcid ]]
    });
    console.log(cids[i]);
}

c013327834
c000566172


この時点では、親コントラクトからサブコントラクトを呼び出すアクセス権が設定されていないため、コントラクトの実行は失敗します。

In [4]:
var resp = await rpc.call(adminWallet, cids[0]);
console.log(resp);
var resp = await rpc.call(adminWallet, cids[1]);
console.log(resp);

{
  txno: 702045,
  txid: 'xwwHPQvzL486BkWcLWRseSQ8XgPYnsXvtAJvkRjurcKZKB',
  status: 'aborted',
  value: 'denied: accessible permission\n    at c013327834:1:16'
}
{
  txno: 702047,
  txid: 'xAGGaTzchJBfrEYkEhaThsRs9LWytzEW6wE3UpLtCRR88',
  status: 'aborted',
  value: 'denied: accessible permission\n    at c000566172:1:16'
}


## アクセス権の設定

サブコントラクトを呼び出すアクセス権をドメインに付与します。  
ドメインに含まれるすべてのコントラクトにアクセス権が付与されます。

In [5]:
await rpc.call(adminWallet, 'c1update', { id: subcid, prop: 'accessible_to', value: [ domain ] });

{
  txno: 702049,
  txid: 'x5uGqkgLLSF59cPCm88nJvp2s22YKBpZzwVb7yoQ34VaEB',
  status: 'ok',
  value: null
}

アクセス権が付与されたため、どの親コントラクトの実行も成功します。

In [6]:
var resp = await rpc.call(adminWallet, cids[0]);
console.log(resp);
var resp = await rpc.call(adminWallet, cids[1]);
console.log(resp);

{
  txno: 702050,
  txid: 'xdgURR6ohbeVeJAbQbqsT4D4oei9qTcgBAjhb9rER9sQKB',
  status: 'ok',
  value: 'c013327834'
}
{
  txno: 702052,
  txid: 'x6x7ZRYtkVywYTRnVn5yS7coa5PLv6kmXPGdGmQUqk5nCB',
  status: 'ok',
  value: 'c000566172'
}


## あとからドメインに作成したコントラクトに対しても、アクセス権の指定は有効

ここで、もうひとつ親コントラクトをドメインに作成します。

In [7]:
    cids[2] = await deploySmartContract(function(){
        return openContract('__subcid__').call();
    }, {
        name: 'adv7cnt2',
        replacers: [['__subcid__', subcid ]]
    });

'c013016093'

アクセス権がすでに付与されているため、あらたに作成した親コントラクトの実行も成功します。

In [8]:
var resp = await rpc.call(adminWallet, cids[2]);
console.log(resp);

{
  txno: 702057,
  txid: 'xYpPRBZTifKiyHHg8zdNwW3xkdGTfUtNkswJQMV6SLUoCB',
  status: 'ok',
  value: 'c013016093'
}
